# NZEB Climate-Aware Transfer Learning
This notebook runs the experiments for the research paper *Quantifying Climate-Aware Transferability in Building Energy Prediction*.


## Phase 0: Setup
Ensure the `src` directory is accessible (either uploaded to Google Drive or in the same directory as this notebook).


In [ ]:
import sys
import os

# If running on Colab, we mount drive and append src to path
try:
    import google.colab
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')

    # Auto-find the shortcut in your MyDrive
    base_drive = '/content/drive/MyDrive'
    SRC_PATH = None
    for folder in os.listdir(base_drive):
        potential_path = os.path.join(base_drive, folder)
        try:
            if os.path.isdir(potential_path) and 'src' in os.listdir(potential_path):
                SRC_PATH = potential_path
                break
        except:
            continue # Skip inaccessible folders
    
    if not SRC_PATH:
        print('WARNING: Could not automatically find the project folder shortcut in MyDrive.')
        print('Please manually check the folder name in your Google Drive and set SRC_PATH.')
    else:
        print(f'✅ Found project shortcut path: {SRC_PATH}')
        if SRC_PATH not in sys.path:
            sys.path.append(SRC_PATH)
        os.chdir(SRC_PATH)
except ImportError:
    IN_COLAB = False

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from src import config, data_loader, feature_engineering, metrics, model, transfer, visualization

print('Setup complete!')


## Phase 1: Data Loading & Exploration
Load pre-merged dataset from Drive and validate.


In [ ]:
# Download the data folder from the shared Google Drive link
!pip install -U gdown -q
!gdown --folder https://drive.google.com/drive/folders/1JRzgEjmTM_Kbiht9GPEtSGHIvqZo3HO3?usp=sharing -O data/

# Set the path to the merged CSV (Update the filename if it differs in the folder)
csv_path = 'data/merged_data.csv'


try:
    df = data_loader.load_merged_data(csv_path)
    # Filter to electricity only
    df = df[df['meter'] == 0].copy()
    data_loader.validate_data(df, expected_sites=list(config.SITES.keys()))
    
    site_dfs = data_loader.split_by_site(df)
    print("Available sites:", site_dfs.keys())
    
    # Exploration
    for site_id, site_df in site_dfs.items():
        print(f"\nSite {site_id} ({config.SITES[site_id]}):")
        print(f"Shape: {site_df.shape}")
        
except FileNotFoundError:
    print(f"Please place your merged data at {csv_path} before continuing.")



## Phase 2: Feature Engineering
Apply temporal features, log transform, and scaling.


In [ ]:
# Process features for all sites
processed_dfs = {}
all_uses = set()

# Get all primary uses across all sites for consistent encoding
if 'df' in locals():
    all_uses = set(df['primary_use'].unique())

X_dict = {}
y_dict = {}
features_list = None

if 'site_dfs' in locals():
    for site_id, site_df in site_dfs.items():
        X, y, f_names = feature_engineering.prepare_features(site_df, all_possible_uses=all_uses)
        X_dict[site_id] = X
        y_dict[site_id] = y
        features_list = f_names # Same for all due to all_possible_uses
        
    print("Features ready:", len(features_list), "features.")



## Phase 3: Baseline Models (5-Fold CV)
Train scratch models for each site to establish baseline performance.


In [ ]:
baseline_results = []

if 'X_dict' in locals():
    for site_id in config.SITES.keys():
        print(f"\nTraining Baseline for Site {site_id} ({config.SITES[site_id]})...")
        X_site = X_dict[site_id]
        y_site = y_dict[site_id]
        
        # We scale inside each CV fold ideally, or scale before splitting (simplified here)
        # Simplified: scale whole site data for baseline baseline evaluation
        X_scaled, _, _ = feature_engineering.scale_features(X_site, X_site)
        
        results = model.train_with_kfold(
            X_scaled, y_site, 
            n_folds=config.N_FOLDS,
            layers=config.ANN_LAYERS,
            learning_rate=config.LEARNING_RATE,
            epochs=config.EPOCHS,
            patience=config.PATIENCE
        )
        
        summary = metrics.summarize_kfold(results)
        summary['site_id'] = site_id
        summary['location'] = config.SITES[site_id]
        summary['model_type'] = 'Scratch'
        baseline_results.append(summary)
        
    df_baselines = pd.DataFrame(baseline_results)
    print(df_baselines[['location', 'R2_mean', 'R2_std', 'MAPE_mean']])



## Phase 4: Transfer Learning
Perform transfer learning for all 6 bidirectional scenarios using 5-fold CV.


In [ ]:
transfer_results = []

if 'X_dict' in locals():
    for scenario in config.TRANSFER_SCENARIOS:
        src = scenario['source']
        tgt = scenario['target']
        print(f"\nTransfer: {config.SITES[src]} -> {config.SITES[tgt]}")
        
        # In a real run, you'd save the source model during baseline training.
        # Here we assume a pre-trained source model exists in models/ dir.
        source_model_path = f"models/{config.SITES[src]}_baseline.keras"
        
        if os.path.exists(source_model_path):
            X_tgt_scaled, _, _ = feature_engineering.scale_features(X_dict[tgt], X_dict[tgt])
            
            t_res = transfer.transfer_with_kfold(
                source_model_path,
                X_tgt_scaled, y_dict[tgt],
                n_folds=config.N_FOLDS,
                n_freeze=config.N_FREEZE_LAYERS,
                lr=config.TRANSFER_LEARNING_RATE,
                epochs=config.EPOCHS,
                patience=config.PATIENCE
            )
            summary = metrics.summarize_kfold(t_res)
            summary['source'] = config.SITES[src]
            summary['target'] = config.SITES[tgt]
            summary['model_type'] = 'Transfer'
            summary['climate_dist'] = config.CLIMATE_DISTANCE_MATRIX[src][tgt]
            transfer_results.append(summary)
        else:
            print(f"Source model not found: {source_model_path}")
            
    if transfer_results:
        df_transfer = pd.DataFrame(transfer_results)
        print(df_transfer[['source', 'target', 'climate_dist', 'R2_mean', 'R2_std']])



## Phase 5: Data Efficiency Experiment
Evaluate transfer vs scratch on varying fractions of target data.


In [ ]:
efficiency_results = []

# Implementation of sampling and comparing goes here...
# This will iterate over config.DATA_FRACTIONS and config.SEEDS.
print("Placeholder for Phase 5 Data Efficiency loop.")



## Phase 6: Transferability Matrix & Climate Distance
Analyze R2 gains against climate distance.


In [ ]:
# Aggregate results into matrix
# visualization.plot_transferability_heatmap(...)
# visualization.plot_climate_distance_vs_gain(...)
print("Placeholder for Phase 6 analysis.")



## Phase 7: Explainability (SHAP)
Identify which features transfer.


In [ ]:
# import shap
# explainer = shap.DeepExplainer(...)
# shap.summary_plot(...)
print("Placeholder for Phase 7 SHAP explainability.")

